### **Import libraries & Load Dataset**


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import io
import numpy as np


print("Please upload your CSV file:")
uploaded = files.upload()

Please upload your CSV file:


Saving healthcare_risk_analysis_dataset.xlsx to healthcare_risk_analysis_dataset.xlsx


In [2]:
# Assuming only one file is uploaded, get the filename
for fn in uploaded.keys():
  file_name = fn

# Read the Excel file into a pandas DataFrame
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

df.head()

,Patient_ID,Patient_Name,Age,Gender,Height_cm,Weight_kg,BMI,Smoking_Status,Alcohol_Consumption,Physical_Activity,...,Family_History,Systolic_BP,Diastolic_BP,Cholesterol_Level,Glucose_Level,Sleep_Hours,Stress_Level,Diagnosed_Condition,Risk_Level,Last_Visit_Date
0,1,Jose Diaz,69,Other,156.9,76.8,31.2,Never,Occasional,Low,...,Yes,137,97,203.0,104.0,6.1,6,Hypertension,High,2024-04-13
1,2,Sherri Armstrong,66,Other,158.8,35.4,14.1,Never,Occasional,Low,...,No,142,84,199.0,71.0,9.2,4,NaN,Moderate,2025-10-04
2,3,Tina Sawyer,81,Male,179.3,70.4,21.9,Former,Occasional,Moderate,...,Yes,128,107,151.0,129.0,8.2,8,Heart Disease,Moderate,2024-04-26
3,4,Charles Christensen,64,Male,157.0,66.2,26.9,Former,Occasional,Low,...,Yes,115,92,195.0,83.0,6.5,4,NaN,Moderate,2025-01-26
4,5,Alex Anderson,71,Male,146.2,56.3,26.4,Never,Occasional,Low,...,No,136,90,238.0,74.0,7.8,5,NaN,Moderate,2023-10-04


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Patient_ID           6000 non-null   int64         
 1   Patient_Name         6000 non-null   object        
 2   Age                  6000 non-null   int64         
 3   Gender               6000 non-null   object        
 4   Height_cm            6000 non-null   float64       
 5   Weight_kg            6000 non-null   float64       
 6   BMI                  5940 non-null   float64       
 7   Smoking_Status       6000 non-null   object        
 8   Alcohol_Consumption  4176 non-null   object        
 9   Physical_Activity    6000 non-null   object        
 10  Diet_Type            6000 non-null   object        
 11  Family_History       6000 non-null   object        
 12  Systolic_BP          6000 non-null   int64         
 13  Diastolic_BP         6000 non-nul

### Data Cleaning: Handling Missing Values

Based on `df.info()`, several columns have missing values. We will impute them as follows:
- **Numerical columns** (`BMI`, `Cholesterol_Level`, `Glucose_Level`, `Sleep_Hours`): Impute with the **median** to be robust to outliers.
- **Categorical columns** (`Alcohol_Consumption`, `Diagnosed_Condition`): Impute with a placeholder value, 'Unknown'.

In [4]:
# Imputing numerical missing values with the median
for col in ['BMI', 'Cholesterol_Level', 'Glucose_Level', 'Sleep_Hours']:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        #print(f"Filled missing values in '{col}' with median: {median_val}")

# Imputing categorical missing values with 'Unknown'
for col in ['Alcohol_Consumption', 'Diagnosed_Condition']:
    if df[col].isnull().any():
        df[col] = df[col].fillna('Unknown')
        #print(f"Filled missing values in '{col}' with 'Unknown'")

print("\nDataFrame info after handling missing values:")
df.info()


DataFrame info after handling missing values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Patient_ID           6000 non-null   int64         
 1   Patient_Name         6000 non-null   object        
 2   Age                  6000 non-null   int64         
 3   Gender               6000 non-null   object        
 4   Height_cm            6000 non-null   float64       
 5   Weight_kg            6000 non-null   float64       
 6   BMI                  6000 non-null   float64       
 7   Smoking_Status       6000 non-null   object        
 8   Alcohol_Consumption  6000 non-null   object        
 9   Physical_Activity    6000 non-null   object        
 10  Diet_Type            6000 non-null   object        
 11  Family_History       6000 non-null   object        
 12  Systolic_BP          6000 non-null   int64 

In [5]:
# DUPLICATE CHECK
print("=" * 60)
print("STEP 4 — DUPLICATE CHECK")
print("=" * 60)

dupes = df.duplicated().sum()
print(f"Duplicate rows  : {dupes}")
print(f"Duplicate IDs   : {df['Patient_ID'].duplicated().sum()}")

STEP 4 — DUPLICATE CHECK
Duplicate rows  : 0
Duplicate IDs   : 0


### **Outlier Check**

#### **Outlier Detection using IQR Method**

We'll use the Interquartile Range (IQR) method to identify outliers in the numerical columns. An outlier is defined as a data point that falls below Q1 - 1.5 * IQR or above Q3 + 1.5 * IQR.

In [7]:
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers.shape[0], lower_bound, upper_bound

numerical_cols = ['Age', 'Height_cm', 'Weight_kg', 'BMI', 'Systolic_BP', 'Diastolic_BP', 'Cholesterol_Level', 'Glucose_Level', 'Sleep_Hours', 'Stress_Level']
outlier_summary = {}

for col in numerical_cols:
    num_outliers, lower_bound, upper_bound = detect_outliers_iqr(df, col)
    if num_outliers > 0:
        outlier_summary[col] = {
            'num_outliers': num_outliers,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound
        }

if outlier_summary:
    print("Outliers detected in the following numerical columns (using IQR method):")
    for col, info in outlier_summary.items():
        print(f"- {col}: {info['num_outliers']} outliers (Values below {info['lower_bound']:.2f} or above {info['upper_bound']:.2f})")
else:
    print("No significant outliers detected in numerical columns using the IQR method.")

Outliers detected in the following numerical columns (using IQR method):
- Height_cm: 44 outliers (Values below 140.80 or above 195.20)
- Weight_kg: 40 outliers (Values below 32.01 or above 112.51)
- BMI: 63 outliers (Values below 9.80 or above 41.80)
- Systolic_BP: 40 outliers (Values below 87.50 or above 171.50)
- Diastolic_BP: 74 outliers (Values below 58.50 or above 110.50)
- Cholesterol_Level: 31 outliers (Values below 132.00 or above 300.00)
- Glucose_Level: 35 outliers (Values below 54.00 or above 166.00)
- Sleep_Hours: 43 outliers (Values below 3.00 or above 11.00)


### **Outlier Removal using IQR Method**

Now, we'll remove these identified outliers from the DataFrame. For each column where outliers were detected, we will filter the DataFrame to include only the rows where the values fall within the calculated lower and upper bounds.

In [13]:
# Make a copy of the DataFrame to perform outlier removal, avoiding SettingWithCopyWarning
df_cleaned = df.copy()

initial_rows = df_cleaned.shape[0]

for col in numerical_cols:
    num_outliers, lower_bound, upper_bound = detect_outliers_iqr(df, col)
    if num_outliers > 0:
        # Filter out outliers for the current column
        df_cleaned = df_cleaned[(df_cleaned[col] >= lower_bound) & (df_cleaned[col] <= upper_bound)]

rows_removed = initial_rows - df_cleaned.shape[0]

print(f"Original DataFrame shape: {df.shape}")
print(f"Cleaned DataFrame shape after outlier removal: {df_cleaned.shape}")
print(f"Total rows removed due to outliers: {rows_removed}")

# Display the first few rows of the cleaned DataFrame
display(df_cleaned.head())

Original DataFrame shape: (6000, 21)
Cleaned DataFrame shape after outlier removal: (5660, 21)
Total rows removed due to outliers: 340


,Patient_ID,Patient_Name,Age,Gender,Height_cm,Weight_kg,BMI,Smoking_Status,Alcohol_Consumption,Physical_Activity,...,Family_History,Systolic_BP,Diastolic_BP,Cholesterol_Level,Glucose_Level,Sleep_Hours,Stress_Level,Diagnosed_Condition,Risk_Level,Last_Visit_Date
0,1,Jose Diaz,69,Other,156.9,76.8,31.2,Never,Occasional,Low,...,Yes,137,97,203.0,104.0,6.1,6,Hypertension,High,2024-04-13
1,2,Sherri Armstrong,66,Other,158.8,35.4,14.1,Never,Occasional,Low,...,No,142,84,199.0,71.0,9.2,4,Unknown,Moderate,2025-10-04
2,3,Tina Sawyer,81,Male,179.3,70.4,21.9,Former,Occasional,Moderate,...,Yes,128,107,151.0,129.0,8.2,8,Heart Disease,Moderate,2024-04-26
3,4,Charles Christensen,64,Male,157.0,66.2,26.9,Former,Occasional,Low,...,Yes,115,92,195.0,83.0,6.5,4,Unknown,Moderate,2025-01-26
4,5,Alex Anderson,71,Male,146.2,56.3,26.4,Never,Occasional,Low,...,No,136,90,238.0,74.0,7.8,5,Unknown,Moderate,2023-10-04


### **Outlier Check**

In [14]:
print("\n" + "=" * 60)
print("OUTLIER Check")
print("=" * 60)

num_cols = ["Age", "Height_cm", "Weight_kg", "BMI", "Systolic_BP",
            "Diastolic_BP", "Cholesterol_Level", "Glucose_Level",
            "Sleep_Hours", "Stress_Level"]

outlier_summary = {}
for col in num_cols:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    outlier_summary[col] = {"Lower Fence": round(lower, 2),
                             "Upper Fence": round(upper, 2),
                             "Outliers": n_out,
                             "Outlier %": round(n_out / len(data) * 100, 2)}

out_df = pd.DataFrame(outlier_summary).T
print(out_df)


OUTLIER Check
                   Lower Fence  Upper Fence  Outliers  Outlier %
Age                     -15.50       116.50       0.0       0.00
Height_cm               140.80       195.20      44.0       0.73
Weight_kg                32.01       112.51      40.0       0.67
BMI                       9.80        41.80      63.0       1.05
Systolic_BP              87.50       171.50      40.0       0.67
Diastolic_BP             58.50       110.50      74.0       1.23
Cholesterol_Level       132.00       300.00      31.0       0.52
Glucose_Level            54.00       166.00      35.0       0.58
Sleep_Hours               3.00        11.00      43.0       0.72
Stress_Level             -4.50        15.50       0.0       0.00


### Data Normalization: Min-Max Scaling

Normalization scales numerical features to a specific range (e.g., 0 to 1). This is often important for machine learning algorithms that are sensitive to the magnitude of feature values. Here, we'll apply Min-Max scaling to the `Age` column as an example:

$X_{normalized} = (X - X_{min}) / (X_{max} - X_{min})$

In [15]:
# Apply Min-Max scaling to the 'Age' column
min_age = df['Age'].min()
max_age = df['Age'].max()
df['Age_normalized'] = (df['Age'] - min_age) / (max_age - min_age)

print("\nFirst 5 rows of DataFrame with 'Age_normalized' column:")
display(df[['Age', 'Age_normalized']].head())


First 5 rows of DataFrame with 'Age_normalized' column:


,Age,Age_normalized
0,69,0.772727
1,66,0.727273
2,81,0.954545
3,64,0.696970
4,71,0.803030
